# THIS IS S3_INGESTION CODE - MODULAR N CONFIGURATION DONE

In [0]:
# I will start from here ----- s3 ingestion ----- homework from uday bhaiya

# here basically we fetch the data from sql server cloud and dump it in raw data folder of databricks

# we used modular architecture in a sense that we created a common functions such as read n write - which will be called for each file

# also we hide our access key using configuration - we did not hardcode it - we used databricks secret scope - to store access keys

# we also created the widgets to pass path of files

# steps are just previous steps , but this time with the use of common utils functions and confugurations - okay


# =======1.IMPORT LIBRARIES & FUNCTIONS=================

import json
from datetime import date
from common_utils.logging import get_logger
import importlib, common_utils.ingestor
importlib.reload(common_utils.ingestor)
from common_utils.ingestor import read_s3_parquet, write_raw

logger = get_logger('s3-ingestion')

# =======2.CREATE WIDGETS FOR PATH=====================
dbutils.widgets.text('path',"")
config_path = dbutils.widgets.get('path')

# =======3.CONFIG VARIABLES FOR ACCESSING KEYS USING CONFIG JSON FILE ===========

with open(config_path,'r') as f:
    config = json.load(f)

source_config = config['source']
target_config = config['target']
write_options_target = config['write_options']

print(source_config)


# here we have already created the secret scope inside the databricks n used the terminal to see it

aws_access_key_id = dbutils.secrets.get(scope = 'retail-platform-dev', key = source_config['aws_access_key_id'])
print(f'our secret access id is hidden as: {aws_access_key_id}')
aws_secret_access_key = dbutils.secrets.get(scope = 'retail-platform-dev', key= source_config['aws_secret_access_key'])
print(f'our secret aws access key is hidden as : {aws_secret_access_key}')

# ========4.FETCH DATA FROM SQL SERVER & READ IT USING READ_JDBC FUNCTION=================

df = read_s3_parquet(spark,aws_access_key_id, aws_secret_access_key, source_config['path'])
logger.info('read %s rows', df.count())
logger.info('sample data looks like.....')
df.show()


# ========5.WRITING THE DATA INTO RAW DATA FOLDER WITH CURRENT DATE FOLDER ====================

from datetime import date
run_date = date.today().isoformat()
logger.info('load date is %s', run_date)

target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"
logger.info("writiing data to %s", target_path)

target_path = write_raw(df, target_path,target_config["file_format"], target_config["mode"], None)
logger.info("data landed at %s", target_path)



# CODE ABOVE WORKS PERFECTLY FINE